# Training + comparison experiments (parameterized)

This notebook runs a **small experimental grid** where the only differences between models are explicit toggles:

- topology mode: padded layered (feedforward BINN constraint) vs native DAG (no padding)
- convolution rule: unique edge weights vs shared weights vs gated weights
- latent dimensionality: scalar nodes vs vector embeddings

For rigorous claims, run this notebook with multiple seeds (and ideally repeated CV). The code below is written so you can scale that up by changing a small config section.


In [ ]:
# ==== 0) Setup: paths + run config ====
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

BASE_DIR = Path(os.environ.get("BINN_GNN_BASE", ".")).resolve()
OUT_DIR  = BASE_DIR / "outputs"
RAW_GRAPH_DIR = OUT_DIR / "graph"
LAYERED_GRAPH_DIR = OUT_DIR / "graph_layered_binn"

EXPR_PARQUET = OUT_DIR / "expr_reactome_tcga_tumor_normal.parquet"
Y_CSV        = OUT_DIR / "y_tcga_tumor_normal.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Experiment knobs
SEEDS = [0, 1, 2]         # increase for stronger conclusions
VAL_FRAC = 0.2
EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.2
D_VECTOR = 16

# Sanity checks
required = [EXPR_PARQUET, Y_CSV, RAW_GRAPH_DIR / "node_table.csv", RAW_GRAPH_DIR / "edge_index.pt",
            LAYERED_GRAPH_DIR / "node_table_layered.csv", LAYERED_GRAPH_DIR / "edge_index_layered.pt"]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required artifacts:\n" + "\n".join(missing))

In [ ]:
# ==== 1) Load dataset (X_gene, y) ====
expr = pd.read_parquet(EXPR_PARQUET)   # genes x samples
y_df = pd.read_csv(Y_CSV, index_col=0)

samples = y_df.index.astype(str).tolist()
y_np = y_df.iloc[:,0].to_numpy(dtype=np.int64)

# Align expression columns to label order
expr = expr.loc[:, samples]

X_gene = torch.tensor(expr.to_numpy(dtype=np.float32).T)  # [S, G]
y = torch.tensor(y_np, dtype=torch.long)

print("X_gene:", tuple(X_gene.shape), "y:", tuple(y.shape), "tumor_frac:", float(y.float().mean()))

In [ ]:
# ==== 2) Build schedules and gene maps (layered + raw DAG) ====
import numpy as np
import pandas as pd
import torch

from binn_gnn.graph.schedule import build_layered_schedule, build_depth_schedule

# Layered (padded) graph schedule
node_layered = pd.read_csv(LAYERED_GRAPH_DIR / "node_table_layered.csv")
edge_layered = pd.read_csv(LAYERED_GRAPH_DIR / "edge_table_layered.csv")
edge_index_layered = torch.load(LAYERED_GRAPH_DIR / "edge_index_layered.pt").long()

layered_schedule = build_layered_schedule(
    edge_index=edge_index_layered,
    node_layer=node_layered["layer"].to_numpy(dtype=np.int64),
    edge_type=edge_layered["edge_type"].astype(str).tolist(),
    gene_ids=node_layered.query("node_type=='gene' and layer==0")["layered_id"].astype(int).tolist(),
    root_ids=torch.load(LAYERED_GRAPH_DIR / "root_pathway_idx.pt").long().tolist() if (LAYERED_GRAPH_DIR / "root_pathway_idx.pt").exists() else [],
)

# Map expression gene order -> layered gene ids
gene_rows = node_layered.query("node_type=='gene' and layer==0")[["orig_name","layered_id"]]
gene_to_layered = dict(zip(gene_rows["orig_name"].astype(str), gene_rows["layered_id"].astype(int)))
gene_map_layered = torch.tensor([gene_to_layered[g] for g in expr.index.astype(str)], dtype=torch.long)

print("Layered schedule | N:", layered_schedule.N, "E:", layered_schedule.E, "L:", layered_schedule.L)

# Raw DAG schedule
node_raw = pd.read_csv(RAW_GRAPH_DIR / "node_table.csv")
edge_raw = torch.load(RAW_GRAPH_DIR / "edge_index.pt").long()
edge_table_raw = pd.read_csv(RAW_GRAPH_DIR / "edge_table.csv")

# genes are the first num_genes nodes (as constructed in Phase 1)
num_genes = int((node_raw["node_type"]=="gene").sum())
gene_ids_raw = list(range(num_genes))

# root pathways in the raw pathway-only subgraph: dst not in src (child->parent edges)
pw = edge_table_raw.query("edge_type=='pathway_to_parent'")[["src","dst"]].astype(int)
root_candidates = sorted(set(pw["dst"].tolist()) - set(pw["src"].tolist()))
root_ids_raw = root_candidates

# Map expression gene order -> raw gene ids (raw genes were sorted in construction)
raw_gene_names = node_raw.query("node_type=='gene'")["node_name"].astype(str).tolist()
raw_gene_to_id = {g:i for i,g in enumerate(raw_gene_names)}
gene_map_raw = torch.tensor([raw_gene_to_id[g] for g in expr.index.astype(str)], dtype=torch.long)

depth_schedule = build_depth_schedule(edge_index=edge_raw, gene_ids=gene_ids_raw, root_ids=root_ids_raw)

print("Raw DAG schedule | N:", depth_schedule.N, "E:", depth_schedule.E, "D:", depth_schedule.D, "roots:", len(root_ids_raw))

In [ ]:
# ==== 3) Define model constructors (the experimental grid) ====
from binn_gnn.models.layered import BINNExactD1, GCNSharedD1, GATGateD1, VectorSharedD, DenseMLPBaseline
from binn_gnn.models.dag import DAGBINNExactD1

def make_models():
    return {
        # padded layered (feedforward BINN semantics)
        "layered_binn_exact_d1": lambda: BINNExactD1(schedule=layered_schedule, gene_map=gene_map_layered, dropout=DROPOUT, learn_padding=False),
        "layered_gcn_shared_d1": lambda: GCNSharedD1(schedule=layered_schedule, gene_map=gene_map_layered, dropout=DROPOUT),
        "layered_gat_gate_d1":   lambda: GATGateD1(schedule=layered_schedule, gene_map=gene_map_layered, dropout=DROPOUT),
        "layered_vector_shared_d16": lambda: VectorSharedD(schedule=layered_schedule, gene_map=gene_map_layered, d=D_VECTOR, dropout=DROPOUT),

        # native DAG (no copy-node padding)
        "dag_binn_exact_d1": lambda: DAGBINNExactD1(schedule=depth_schedule, gene_map=gene_map_raw, dropout=DROPOUT),

        # dense baseline
        "dense_mlp": lambda: DenseMLPBaseline(in_dim=X_gene.shape[1], dropout=DROPOUT),
    }

In [ ]:
# ==== 4) Training loop (repeated seeds) ====
from torch.utils.data import DataLoader, TensorDataset
from binn_gnn.experiments.train import set_seed, train_classifier

def run_one_seed(seed):
    set_seed(seed)
    idx = np.arange(len(y_np))
    train_idx, val_idx = train_test_split(idx, test_size=VAL_FRAC, random_state=seed, stratify=y_np)

    X_train = X_gene[train_idx]
    y_train = y[train_idx]
    X_val   = X_gene[val_idx]
    y_val   = y[val_idx]

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=256, shuffle=False)

    results = []
    for name, ctor in make_models().items():
        model = ctor()
        r = train_classifier(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            epochs=EPOCHS,
            lr=LR,
            weight_decay=WEIGHT_DECAY,
            early_stopping_patience=max(3, EPOCHS//2),
            verbose=False,
        )
        row = {"seed": seed, "model": name, **r.val_metrics}
        results.append(row)
        print(f"seed={seed} | {name} | {r.val_metrics}")
    return results

all_rows = []
for s in SEEDS:
    all_rows.extend(run_one_seed(s))

df = pd.DataFrame(all_rows)
df

In [ ]:
# ==== 5) Aggregate results ====
import pandas as pd

summary = (
    df.groupby("model")
      .agg(["mean","std"])
      .sort_values(("roc_auc","mean"), ascending=False)
)

summary